Color–Magnitude Diagram

A Jupyter notebook that plots colour–magnitude diagrams (CMD) from multiple stellar databases.

**Data sources**
- **HYG v4.1** – Hipparcos/Yale/Gliese stellar catalog (B–V colour index `ci`, absolute visual magnitude `absmag`)
- **Pleiades** – Gaia BP-RP colour and G-band magnitude (computed absolute magnitude)
- **47 Tuc** – Globular cluster (Gaia DR3, xVilas/globular-cluster-age-47tuc) – BP-RP and G-band magnitude

**Features**
1. `load_hyg()` loads the HYG catalog.
2. `load_pleiades()` loads and computes Pleiades data.
3. `load_47tuc()` loads and computes 47 Tuc data (fixed distance).
4. `plot_cmd()` creates a CMD plot.
5. Side-by-side histogram of absolute magnitudes (inverted x-axis).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Activate interactive backend
%matplotlib widget

def load_hyg(csv_url):
    """Load HYG v4.1 catalog (has ci and absmag columns)."""
    raw = pd.read_csv(csv_url)
    cols = [c.lstrip('#') for c in raw.columns]
    raw = raw[cols]
    raw = raw.dropna(subset=['ci', 'absmag'])
    return raw

def load_pleiades(csv_url):
    """Load Pleiades Gaia data and compute ci/absmag."""
    raw = pd.read_csv(csv_url)
    raw.columns = [c.lstrip('#') for c in raw.columns]
    raw = raw.dropna(subset=['bp_rp', 'phot_g_mean_mag', 'parallax'])
    raw = raw[raw['parallax'] > 0]
    raw['ci'] = raw['bp_rp']
    raw['absmag'] = raw['phot_g_mean_mag'] - 5 * (np.log10(1000 / raw['parallax']) - 1)
    return raw

def load_47tuc(csv_url):
    """Load globular cluster (47 Tuc) Gaia data and compute ci/absmag using median parallax."""
    raw = pd.read_csv(csv_url, on_bad_lines="skip")
    raw.columns = [c.lstrip('#') for c in raw.columns]
    # VizieR column names may differ - try common variants
    # Columns: bp_rp, phot_g_mean_mag, plx (parallax)
    rename_map = {}
    for col in raw.columns:
        if col.upper() in ('BP-RP', 'BP_RP', 'BPRP'):
            rename_map[col] = 'bp_rp'
        if col.upper() in ('GMAG', 'G MAG'):
            rename_map[col] = 'phot_g_mean_mag'
        if col.upper() in ('PLX', 'PARALLAX'):
            rename_map[col] = 'plx'
    raw = raw.rename(columns=rename_map)
    required = ['bp_rp', 'phot_g_mean_mag', 'plx']
    raw = raw.dropna(subset=required)
    raw = raw[raw['plx'] > 0]
    raw['ci'] = raw['bp_rp']
    median_plx = raw['plx'].median()
    dm = 5 * (np.log10(1000 / median_plx) - 1)
    raw['absmag'] = raw['phot_g_mean_mag'] - dm
    return raw

def plot_cmd(df, title="Colour–Magnitude Diagram"):
    """Plot a Colour–Magnitude Diagram (CMD)."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df['ci'], df['absmag'], alpha=0.5, s=20)
    ax.set_xlabel('B–V colour index (ci)')
    ax.set_ylabel('Absolute visual magnitude (M_V)')
    ax.set_title(title)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


In [ ]:
# --- HYG v4.1 stellar catalog ---
hyg_url = (
    'https://raw.githubusercontent.com/astronexus/HYG-Database/'
    'main/hyg/CURRENT/hygdata_v41.csv'
)
df_hyg = load_hyg(hyg_url)
plot_cmd(df_hyg, 'HYG v4.1 Colour–Magnitude Diagram')
print(f'HYG stars plotted: {len(df_hyg)}')

In [ ]:
# --- Pleiades (Gaia BP-RP / G-band) ---
pleiades_url = (
    'https://raw.githubusercontent.com/AngeloPellizzari/Pleiades-Cluster/'
    'master/Pleiades.csv'
)
df_pleiades = load_pleiades(pleiades_url)
plot_cmd(df_pleiades, 'Pleiades Cluster Colour–Magnitude Diagram')
print(f'Pleiades stars plotted: {len(df_pleiades)}')

In [ ]:
# --- 47 Tuc Globular Cluster (Gaia DR3) ---
tuc_url = "https://raw.githubusercontent.com/xVilas/globular-cluster-age-47tuc/main/data/processed/clean_data.csv"
df_47tuc = load_47tuc(tuc_url)
plot_cmd(df_47tuc, '47 Tuc Globular Cluster Colour–Magnitude Diagram')
print(f'47 Tuc stars plotted: {len(df_47tuc)}')

In [ ]:
# --- Histogram: HYG vs Pleiades absolute magnitudes ---
plt.figure(figsize=(10, 5))
plt.hist([df_hyg['absmag'], df_pleiades['absmag']], 
         bins=30, alpha=0.7, label=['HYG', 'Pleiades'], color=['blue', 'orange'])
plt.xlabel('Absolute Visual Magnitude (M_V)')
plt.ylabel('Count')
plt.legend()
plt.gca().invert_xaxis()  # Brighter (more negative) magnitudes on the left
plt.title('Absolute Magnitude Distribution: HYG vs Pleiades')
plt.tight_layout()
plt.show()

print('Done.')